## XGBoost, LightGBM, CatBoost

- theory behind all 3 models are based on Gradient Boosting with some extensions
- many useful programming functionalities were added that:
    - reduce model variance further (better cross-validation and test performance)
    - make the runtime faster (more efficient implementation)

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from xgboost import XGBRegressor, XGBClassifier

In [2]:
trainf = pd.read_csv("/Users/vaibhavrangan/Downloads/Stat_303-3/Datasets/Car_features_train.csv")
trainp = pd.read_csv("/Users/vaibhavrangan/Downloads/Stat_303-3/Datasets/Car_prices_train.csv")
train = pd.merge(trainf, trainp)

testf = pd.read_csv("/Users/vaibhavrangan/Downloads/Stat_303-3/Datasets/Car_prices_test.csv")
testp = pd.read_csv("/Users/vaibhavrangan/Downloads/Stat_303-3/Datasets/Car_features_test.csv")
test = pd.merge(testf, testp)

predictors = ['mpg', 'engineSize', 'year', 'mileage']
target = 'price'
X_train = train[predictors]
y_train = train[target]
X_test = test[predictors]
y_test = test[target]

- XGBoost is faster than Gradient Boosting because:
    - parallel processing of the NODES OF EACH TREE
    - "histogram features": while finding the decision rule at each node:
        - take a predictor, create a histogram of its values
        - find a threshold using these categorized values
        - this method returns very comparable performance but much faster

- XGBoost performs better than Gradient Boosting because of the extra hyperparameters:
    - will be introduced in the model inputs
    - strongly force the model to avoid overfitting
    - make the model ignore outliers as much as possible

- Does XGBoost miss anything? Yes, Huber loss:
    - Gradient Boosting implements Huber loss very well
    - XGBoost has "pseudo Huber loss," not implemented well
    - just use MAE in XGBoost

# Model Inputs

In [4]:
model = XGBRegressor(
    random_state = 12,
    max_depth = 6, # identical to AdaBoost and Gradient Boosting
    learning_rate = 0.1, # same as AdaBoost and Gradient Boosting
    subsample = 0.8, # same as AdaBoost and Gradient Boosting

    # the extra hyperparameters
    reg_lambda = 1, # the factor multiplied with the Ridge penalty added to the cost function of each tree
    # the ridge penalty consists of "leaf weights" -- measuring the important of a leaf in the tree
        # proportional to the MSE/Gini it reduces, inversely proportional to the complexity it adds
    
    gamma = 0.1,# "tree pruning" hyperparameter, cancels/prunes teh relatively unnecessary leaves by checking their leaf weights and eliminating the leaves with weights below gamma

    # assume we have 20 predictors
    colsample_bytree = 0.5, # a feature/predictor subset that each tree sees, tries to break any excess correlation between trees
    colsample_bylevel = 0.5, # 0.5 here (with 0.5 above) means each level sees 5 (out of 10)
    colsample_bynode = 0.5 # 0.5 here (with 0.5 both above) means each node sees 2 (out of 5)
    # Using all three (or two out of three) is usually too much, unless the data is too high-dim

)
# a group of three hyperparams:
# colsample_bytree: the predictor subset size seen by each tree
# colsample_byLevel: the predictor subset size seen by each level of the tree
# colsample_bynode: the predictor subset size seen by each node of the tree

model.fit(X_train, y_train)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=0.5, colsample_bynode=0.5, colsample_bytree=0.5,
             device=None, early_stopping_rounds=None, enable_categorical=False,
             eval_metric=None, feature_types=None, feature_weights=None,
             gamma=0.1, grow_policy=None, importance_type=None,
             interaction_constraints=None, learning_rate=0.1, max_bin=None,
             max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=6, max_leaves=None,
             min_child_weight=None, missing=nan, monotone_constraints=None,
             multi_strategy=None, n_estimators=None, n_jobs=None,
             num_parallel_tree=None, ...)

# Feature Importances

In [5]:
model.feature_importances_

array([0.19702055, 0.41862735, 0.15708165, 0.22727048], dtype=float32)

# One Extra Tool with .fit()

In [6]:
# let's split test data further

from turtle import mode
from sklearn.model_selection import train_test_split

X_test_sub1, X_test_sub2, y_test_sub1, y_test_sub2 =  train_test_split(X_test, y_test, 
                                                                       test_size = 0.2, random_state = 1)

# Create a new model with a very large n_estimators
model = XGBRegressor(
    random_state = 12,
    max_depth = 6, 
    n_estimators = 20000, # A very large number here!
    early_stopping_rounds = 250
)

# Use the first test subset as validation while training
# As trees are added to the model, the RMSE is calculated with the eval_set data
    # If the RMSE keeps increasing for "early_stopping_rounds" trees, the training stops
# This is a good way to find a good ballpark to start tuning n_estimators
model.fit(X_train, y_train, eval_set = [(X_test_sub1, y_test_sub1)])

[0]	validation_0-rmse:12652.38347
[1]	validation_0-rmse:10349.28768
[2]	validation_0-rmse:8943.21245
[3]	validation_0-rmse:7935.07182
[4]	validation_0-rmse:7303.62198
[5]	validation_0-rmse:6920.79224
[6]	validation_0-rmse:6716.63677
[7]	validation_0-rmse:6555.93199
[8]	validation_0-rmse:6449.74061
[9]	validation_0-rmse:6413.44043
[10]	validation_0-rmse:6339.80850
[11]	validation_0-rmse:6269.05894
[12]	validation_0-rmse:6208.88878
[13]	validation_0-rmse:6192.57314
[14]	validation_0-rmse:6188.92510
[15]	validation_0-rmse:6175.80603
[16]	validation_0-rmse:6173.80931
[17]	validation_0-rmse:6184.71051
[18]	validation_0-rmse:6184.06385
[19]	validation_0-rmse:6203.56047
[20]	validation_0-rmse:6190.15770
[21]	validation_0-rmse:6170.14933
[22]	validation_0-rmse:6150.96807
[23]	validation_0-rmse:6154.70421
[24]	validation_0-rmse:6156.08104
[25]	validation_0-rmse:6162.85975
[26]	validation_0-rmse:6160.50823
[27]	validation_0-rmse:6141.67600
[28]	validation_0-rmse:6157.66531
[29]	validation_0-rmse

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, device=None, early_stopping_rounds=250,
             enable_categorical=False, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=None, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=6,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=20000,
             n_jobs=None, num_parallel_tree=None, ...)

# Tuning the Model

In [7]:
model = XGBRegressor(
    random_state = 12,
    objective = 'reg:squarederror' # This is default, perfectly fine to use this
)

# You can (but usually don't need to) use all inputs discussed above in the grid
# There are rules of thumb for values you can use in such an expensive grid
grid = {
    # The four original hyperparams need to be in the grid
    'max_depth': [4,6,8], # These are shown to be good values for Gradient Boosting models
    'n_estimators': [200, 300, 400], # This needs to be tuned -- no tricks
    'learning_rate': [0.01, 0.1, 1], # Different orders of magnitude is fine.
    'subsample': [0.75, 1], # Use a couple of different ratios

    # Among these two hyperparams, tuning one can be enough
    'reg_lambda': [0.01, 0.1, 1], # Different orders of magnitude is fine.
    'gamma': [0, 0.1], # If tuning gamma, make 0 an option

    # Among the colsample inputs, just tune one of them
    'colsample_bytree': [0.5, 1.0]
}

# Classifier


- XGBClassifier has all the inputs as XGBRegressor, except for one more important input
- scale_pos_weight: assigns higher weights to Class 1 observations – makes Class 1 observations more important to predict
- this always improves recall, accuracy and precision may increase/decrease
- increasing scale_pos_weight too much risks disregarding Class 0 observations too much

In [9]:
model = XGBClassifier(
    random_state = 12,
    objective = 'binary:logistic',

    # Same inputs as the regressor
    max_depth = 6, 
    n_estimators = 20, 
    learning_rate = 0.1,
    subsample = 0.8, 

    # The extra hyperparameters -- the ones that mitigate overfitting
    reg_lambda = 1, #
    gamma = 0.1,
    colsample_bytree = 0.5,  
    colsample_bylevel = 0.5, 
    colsample_bynode = 0.5, 

    # The extra classifier input
    scale_pos_weight = 4 # Class 1 obs are 4 times more important as Class 0 obs
    # scale_pos_weight should be fixed in the model, not tuned
# A good value to fix is # Class 0 obs/# Class 1 obs in the data
)